# Silver Layer: Cleaned & Conformed Tables

This notebook is the **silver** stage of the NEM medallion pipeline. It reads the three bronze tables (everything stored as `STRING`, unclean, untyped) and rebuilds each one as a cleaned, typed, deduplicated silver table.

Per the project's Python-for-ingestion / SQL-for-transformation split, all transformation logic here is plain SQL. Each section below reads one bronze table, applies the cleanup rules identified during EDA (see `02_EDA.ipynb`), and writes the result with `CREATE OR REPLACE TABLE`. No rows are ever dropped, values that can't be trusted are cast to `NULL` and flagged with a companion `is_x_missing` boolean rather than silently discarded or imputed. The only rows removed are genuine duplicates on each table's natural key, kept via a `ROW_NUMBER()` tie-break.

Outputs:
- `nem_project.`2_silver`.facilities`
- `nem_project.`2_silver`.facility_power_emissions`
- `nem_project.`2_silver`.region_price_demand`

## 0. Setup

In [0]:
%sql
USE CATALOG nem_project;

## 1. Facilities → `nem_project.2_silver.facilities`

Cleanup applied, per the findings in `02_EDA.ipynb`:
- Drop `network_id` (constant value across every row, no analytical use).
- Strip the trailing `"1"` suffix from `network_region` (e.g. `NSW1` → `NSW`).
- Strip the `<p>` / `</p>` tags wrapping every `facility_description`.
- Lowercase `dispatch_type`.
- Create `fueltech_label` (readable name) and `fueltech_category` (coarser grouping) from `fueltech_id`.
- Create `is_location_missing` when latitude or longitude is null, `data_is_missing` when `data_first_seen` or `data_last_seen` is null.
- Deduplicate on the natural key (`facility_code`, `unit_code`), keeping the row with the most recent `data_last_seen`.

In [0]:
%sql
-- ===========================================================================
-- Build silver.facilities from bronze.facilities
-- -------------------------------------------------------------------------
-- Purpose: Read the raw, untyped bronze facilities table and produce a
--          cleaned, typed, deduplicated silver table.
--
-- Steps:
--   1. Deduplicate rows on the natural key (facility_code, unit_code),
--      keeping the row with the most recent data_last_seen.
--   2. Cast every column from STRING to its proper type.
--   3. Clean specific columns (network_region, facility_description,
--      dispatch_type).
--   4. Derive human-readable fueltech_label and a coarser fueltech_category
--      from the raw fueltech_id.
--   5. Add boolean flags for missing coordinates and missing data timestamps
--      rather than dropping the rows.
-- ===========================================================================

CREATE OR REPLACE TABLE nem_project.`2_silver`.facilities AS
WITH deduped AS (
  SELECT
    *,
    -- Number rows within each (facility_code, unit_code) group, newest first.
    -- We keep only rn = 1 below, so duplicates are discarded.
    ROW_NUMBER() OVER (
      PARTITION BY facility_code, unit_code
      ORDER BY data_last_seen DESC
    ) AS rn
  FROM nem_project.`1_bronze`.facilities
)
SELECT
  -- Natural key columns: cast to STRING for consistent typing
  CAST(facility_code AS STRING) AS facility_code,
  CAST(facility_name AS STRING) AS facility_name,

  -- Strip the trailing "1" suffix (NSW1 -> NSW)
  CAST(regexp_replace(network_region, '1$', '') AS STRING) AS network_region,

  -- Remove wrapping <p> / </p> tags left over from HTML source data
  CAST(regexp_replace(facility_description, '</p>|<p>', '') AS STRING) AS facility_description,

  CAST(unit_code AS STRING) AS unit_code,
  CAST(fueltech_id AS STRING) AS fueltech_id,

  -- fueltech_label: readable name for each fueltech_id value: falls back to the raw fueltech_id if no mapping matches
  CAST(
    CASE fueltech_id
      WHEN 'battery' THEN 'Battery'
      WHEN 'battery_charging' THEN 'Battery (charging)'
      WHEN 'battery_discharging' THEN 'Battery (discharging)'
      WHEN 'bioenergy_biogas' THEN 'Bioenergy (Biogas)'
      WHEN 'bioenergy_biomass' THEN 'Bioenergy (Biomass)'
      WHEN 'coal_black' THEN 'Coal (Black)'
      WHEN 'coal_brown' THEN 'Coal (Brown)'
      WHEN 'distillate' THEN 'Distillate'
      WHEN 'gas_ccgt' THEN 'Gas (CCGT)'
      WHEN 'gas_ocgt' THEN 'Gas (OCGT)'
      WHEN 'gas_recip' THEN 'Gas (Reciprocating)'
      WHEN 'gas_steam' THEN 'Gas (Steam)'
      WHEN 'gas_wcmg' THEN 'Gas (Waste Coal Mine)'
      WHEN 'hydro' THEN 'Hydro'
      WHEN 'pumps' THEN 'Pumps'
      WHEN 'solar_utility' THEN 'Solar'
      WHEN 'wind' THEN 'Wind'
      ELSE fueltech_id
    END AS STRING
  ) AS fueltech_label,

  -- fueltech_category: broader grouping of fueltech_id for higher-level: aggregations (e.g. all gas variants roll up to 'Gas')
  CAST(
    CASE
      WHEN fueltech_id LIKE 'battery%' THEN 'Battery'
      WHEN fueltech_id LIKE 'bioenergy%' THEN 'Biomass'
      WHEN fueltech_id = 'coal_black' THEN 'Black coal'
      WHEN fueltech_id = 'coal_brown' THEN 'Brown coal'
      WHEN fueltech_id LIKE 'gas%' THEN 'Gas'
      WHEN fueltech_id = 'hydro' THEN 'Hydro'
      WHEN fueltech_id = 'distillate' THEN 'Liquid Fuel'
      WHEN fueltech_id = 'solar_utility' THEN 'Solar'
      WHEN fueltech_id = 'wind' THEN 'Wind'
      ELSE 'Other'
    END AS STRING
  ) AS fueltech_category,

  CAST(status_id AS STRING) AS status_id,

  -- Lowercase dispatch_type for consistency
  CAST(LOWER(dispatch_type) AS STRING) AS dispatch_type,

  -- Geographic coordinates and capacity fields: cast to DOUBLE
  CAST(lat AS DOUBLE) AS lat,
  CAST(lng AS DOUBLE) AS lng,
  CAST(capacity_registered AS DOUBLE) AS capacity_registered,
  CAST(capacity_maximum AS DOUBLE) AS capacity_maximum,
  CAST(capacity_storage AS DOUBLE) AS capacity_storage,

  -- Data lineage timestamps: cast to TIMESTAMP
  CAST(data_first_seen AS TIMESTAMP) AS data_first_seen,
  CAST(data_last_seen AS TIMESTAMP) AS data_last_seen,

  -- Quality flags: surface missing values as booleans instead of dropping rows
  (lat IS NULL OR lng IS NULL) AS is_location_missing,
  (data_first_seen IS NULL OR data_last_seen IS NULL) AS data_is_missing

FROM deduped
-- Keep only the latest row per (facility_code, unit_code) natural key
WHERE rn = 1;

In [0]:
%sql
-- Row count
SELECT COUNT(*) AS row_count
FROM nem_project.`2_silver`.facilities;

In [0]:
%sql
-- Null check sanity query
SELECT
  SUM(CASE WHEN facility_code IS NULL THEN 1 ELSE 0 END) AS null_facility_code,
  SUM(CASE WHEN unit_code IS NULL THEN 1 ELSE 0 END) AS null_unit_code,
  SUM(CASE WHEN network_region IS NULL THEN 1 ELSE 0 END) AS null_network_region,
  SUM(CASE WHEN fueltech_label IS NULL THEN 1 ELSE 0 END) AS null_fueltech_label,
  SUM(CASE WHEN fueltech_category IS NULL THEN 1 ELSE 0 END) AS null_fueltech_category,
  SUM(CASE WHEN is_location_missing THEN 1 ELSE 0 END) AS flagged_missing_location,
  SUM(CASE WHEN data_is_missing THEN 1 ELSE 0 END) AS flagged_missing_data
FROM nem_project.`2_silver`.facilities;

## 2. Facility Power & Emissions → `nem_project.2_silver.facility_power_emissions`

Cleanup applied:
- `time`: cast to `TIMESTAMP`, preserving the original granularity from the source.
- `power` / `emissions`: the bronze columns hold the literal string `"null"` for missing readings (found during EDA), so `NULLIF(..., 'null')` converts those to real `NULL` before casting to `DOUBLE`.
- Set `is_power_missing` / `is_emissions_missing` when the cleaned value is null.
- Deduplicate on the natural key (`facility_code`, `unit_code`, `time`).

In [0]:
%sql
-- ===========================================================================
-- Build silver.facility_power_emissions from bronze.facility_power_emissions
-- -------------------------------------------------------------------------
-- Purpose: Read the raw, untyped bronze facility power & emissions table and
--          produce a cleaned, typed, deduplicated silver table.
--
-- Steps:
--   1. Deduplicate rows on the natural key (facility_code, unit_code, time),
--      keeping the row with the most recent time.
--   2. Cast every column from STRING to its proper type.
--   3. Cast `time` to TIMESTAMP, preserving the original granularity from the source.
--   4. Convert the literal string 'null' found in bronze `power`/`emissions`
--      columns to real NULL via NULLIF before casting to DOUBLE.
--   5. Add boolean flags for missing power and missing emissions rather than
--      dropping the rows.
-- ===========================================================================

CREATE OR REPLACE TABLE nem_project.`2_silver`.facility_power_emissions AS
WITH deduped AS (
  SELECT
    *,
    -- Number rows within each (facility_code, unit_code, time) group,
    -- newest first. We keep only rn = 1 below, so duplicates are discarded.
    ROW_NUMBER() OVER (
      PARTITION BY facility_code, unit_code, time
      ORDER BY time DESC
    ) AS rn
  FROM nem_project.`1_bronze`.facility_power_emissions
),
cleaned AS (
  SELECT
    -- Natural key columns (kept as-is; final cast happens in the outer query)
    facility_code,
    unit_code,
    -- Cast to TIMESTAMP, preserving original granularity for consistency.
    CAST(time AS TIMESTAMP) AS time,
    -- NULLIF(..., 'null') converts the literal string 'null' found in bronze to a real NULL before casting to DOUBLE. Any other non-numeric value also becomes NULL via the cast.
    CAST(NULLIF(power, 'null') AS DOUBLE) AS power,
    CAST(NULLIF(emissions, 'null') AS DOUBLE) AS emissions
  FROM deduped
  -- Keep only the latest row per natural key
  WHERE rn = 1
)
SELECT
  -- Natural key columns: cast to STRING for consistent typing
  CAST(facility_code AS STRING) AS facility_code,
  CAST(unit_code AS STRING) AS unit_code,
  -- Cleaned timestamp
  time,
  -- Typed numeric readings (NULL where the source was 'null' or unparseable)
  power,
  emissions,
  -- Quality flags: surface missing readings as booleans instead of dropping rows
  (power IS NULL) AS is_power_missing,
  (emissions IS NULL) AS is_emissions_missing
FROM cleaned;

In [0]:
%sql
-- Row count
SELECT COUNT(*) AS row_count
FROM nem_project.`2_silver`.facility_power_emissions;

In [0]:
%sql
-- Null check sanity query
SELECT
  SUM(CASE WHEN facility_code IS NULL THEN 1 ELSE 0 END) AS null_facility_code,
  SUM(CASE WHEN unit_code IS NULL THEN 1 ELSE 0 END) AS null_unit_code,
  SUM(CASE WHEN time IS NULL THEN 1 ELSE 0 END) AS null_time,
  SUM(CASE WHEN is_power_missing THEN 1 ELSE 0 END) AS flagged_missing_power,
  SUM(CASE WHEN is_emissions_missing THEN 1 ELSE 0 END) AS flagged_missing_emissions
FROM nem_project.`2_silver`.facility_power_emissions;

## 3. Region Price & Demand → `nem_project.2_silver.region_price_demand`

Cleanup applied:
- Strip the trailing `"1"` suffix from `network_region` (e.g. `NSW1` → `NSW`).
- `time` cast to `TIMESTAMP`; `price` / `demand` cast to `DOUBLE`, with `NULLIF(..., 'null')` clearing the literal `"null"` strings found in bronze during EDA before casting.
- Deduplicate on the natural key (`network_region`, `time`).

In [0]:
%sql
-- ===========================================================================
-- Build silver.region_price_demand from bronze.region_price_demand
-- -------------------------------------------------------------------------
-- Purpose: Read the raw, untyped bronze region price & demand table and
--          produce a cleaned, typed, deduplicated silver table.
--
-- Steps:
--   1. Deduplicate rows on the natural key (network_region, time), keeping the row with the most recent time.
--   2. Strip the trailing "1" suffix from network_region (e.g. NSW1 -> NSW).
--   3. Cast `time` to TIMESTAMP, preserving original granularity from source.
--   4. Convert the literal string 'null' found in bronze `price`/`demand` columns to real NULL via NULLIF before casting to DOUBLE.
-- ===========================================================================

CREATE OR REPLACE TABLE nem_project.`2_silver`.region_price_demand AS
WITH deduped AS (
  SELECT
    *,
    -- Number rows within each (network_region, time) group, newest first.
    -- We keep only rn = 1 below, so duplicates are discarded.
    ROW_NUMBER() OVER (
      PARTITION BY network_region, time
      ORDER BY time DESC
    ) AS rn
  FROM nem_project.`1_bronze`.region_price_demand
)
SELECT
  -- Strip the trailing "1" suffix (NSW1 -> NSW) and cast to STRING for consistent typing
  CAST(regexp_replace(network_region, '1$', '') AS STRING) AS network_region,
  -- Cast to TIMESTAMP, preserving original granularity for consistency
  CAST(time AS TIMESTAMP) AS time,
  -- NULLIF(..., 'null') converts the literal string 'null' found in bronze to a real NULL before casting to DOUBLE. Any other non-numeric value also becomes NULL via the cast.
  CAST(NULLIF(price, 'null') AS DOUBLE) AS price,
  CAST(NULLIF(demand, 'null') AS DOUBLE) AS demand
FROM deduped
-- Keep only the latest row per (network_region, time) natural key
WHERE rn = 1;

In [0]:
%sql
-- Row count
SELECT COUNT(*) AS row_count
FROM nem_project.`2_silver`.region_price_demand;

In [0]:
%sql
-- Null check sanity query
SELECT
  SUM(CASE WHEN network_region IS NULL THEN 1 ELSE 0 END) AS null_network_region,
  SUM(CASE WHEN time IS NULL THEN 1 ELSE 0 END) AS null_time,
  SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS null_price,
  SUM(CASE WHEN demand IS NULL THEN 1 ELSE 0 END) AS null_demand
FROM nem_project.`2_silver`.region_price_demand;

## 4. Summary

Three silver tables were built from the bronze layer, one per source, all via `CREATE OR REPLACE TABLE` with SQL as the sole transformation language:

#### `facilities`
* Dropped the constant `network_id` column.
* Cleaned `network_region` (trailing `"1"` stripped) and `facility_description` (`<p>` tags stripped).
* Lowercased `dispatch_type`.
* Derived `fueltech_label` and `fueltech_category` from `fueltech_id` via a fixed lookup.
* Added `is_location_missing` and `data_is_missing` flags instead of dropping incomplete rows.
* Deduplicated on (`facility_code`, `unit_code`).

#### `facility_power_emissions`
* Cast `time` to `TIMESTAMP`, preserving original granularity.
* Cleared the literal `"null"` strings in `power`/`emissions` before casting to `DOUBLE`.
* Added `is_power_missing` and `is_emissions_missing` flags.
* Deduplicated on (`facility_code`, `unit_code`, `time`).

#### `region_price_demand`
* Cleaned `network_region` (trailing `"1"` stripped).
* Cleared the literal `"null"` strings in `price`/`demand` before casting to `DOUBLE`.
* Cast `time` to `TIMESTAMP`.
* Deduplicated on (`network_region`, `time`).

No rows were dropped in any of the three tables. Every ambiguous or missing value was cast to `NULL` and where a companion flag was specified, surfaced as an `is_x_missing` boolean rather than silently imputed. The only rows removed were genuine duplicates on each table's natural key. These three tables are now the foundation for any downstream gold-layer aggregation or reporting.